# Inserción continua de datos — MongoDB (`eventos_foro`)

Este notebook simula un flujo **near real-time**: cada 2-5 segundos inserta un documento nuevo en la colección `eventos_foro` de la base `staging_steam` (MongoDB). Logstash, corriendo con el plugin `logstash-input-mongodb`, detecta estos documentos nuevos mediante su propio tracking incremental y los indexa automáticamente en Elasticsearch.

**Requisitos:**
```
pip install pymongo Faker
```

In [1]:
import pymongo
import time
import random
from datetime import datetime, timezone
from faker import Faker

fake = Faker()

## Conexión a MongoDB

In [2]:
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["staging_steam"]
collection = db["eventos_foro"]

## Datos de referencia y generador de eventos

`appid` y `tipo_interaccion` son dominios controlados. `nombre_usuario` y `comentario` se generan con Faker.

La función está escrita como código puro (sin dependencias del entorno del notebook) para poder reutilizarla más adelante en un DAG de Airflow sin modificarla.

In [3]:
APPIDS_MUESTRA = [730, 578080, 570, 105600, 271590, 1172470]
TIPOS_INTERACCION = ["post_creado", "comentario", "upvote", "reporte", "share"]
TIPOS_CON_TEXTO = {"post_creado", "comentario"}


def generar_evento():
    tipo = random.choice(TIPOS_INTERACCION)

    evento = {"appid": random.choice(APPIDS_MUESTRA), "tipo_interaccion": tipo,
              "usuario_id": random.randint(1000, 9999), "nombre_usuario": fake.user_name(),
              "fecha_evento": datetime.now(timezone.utc),
              "comentario": fake.sentence() if tipo in TIPOS_CON_TEXTO else None}

    return evento

## Función de inserción

In [4]:
def insertar_evento(collection, evento):
    collection.insert_one(evento)

## Bucle de simulación (near real-time)

Espera aleatoria entre 2 y 5 segundos entre inserciones, simulando tráfico irregular de un foro real. Detén la celda (interrumpir kernel) para parar la simulación.

In [5]:
print("Iniciando simulación de eventos en MongoDB. Interrumpe el kernel para detener.\n")

contador = 0
try:
    while True:
        evento = generar_evento()
        insertar_evento(collection, evento)
        contador += 1

        detalle = f"tipo={evento['tipo_interaccion']}, appid={evento['appid']}, usuario={evento['nombre_usuario']}"
        if evento["comentario"]:
            detalle += f", comentario=\"{evento['comentario']}\""
        print(f"[{contador}] Insertado en Mongo: {detalle}")

        time.sleep(random.randint(2, 5))
except KeyboardInterrupt:
    print("\nSimulación detenida por el usuario.")
    print(f"Total de documentos en la colección: {collection.count_documents({})}")

Iniciando simulación de eventos en MongoDB. Interrumpe el kernel para detener.

[1] Insertado en Mongo: tipo=comentario, appid=1172470, usuario=tammy67, comentario="Way assume early store dark."
[2] Insertado en Mongo: tipo=share, appid=730, usuario=ellenrandolph
[3] Insertado en Mongo: tipo=post_creado, appid=1172470, usuario=qwhitaker, comentario="Nature hotel reason choose clearly four."
[4] Insertado en Mongo: tipo=reporte, appid=730, usuario=salazargeorge
[5] Insertado en Mongo: tipo=share, appid=105600, usuario=kurtpayne
[6] Insertado en Mongo: tipo=upvote, appid=578080, usuario=teresa83
[7] Insertado en Mongo: tipo=post_creado, appid=271590, usuario=mhernandez, comentario="Republican market operation class."
[8] Insertado en Mongo: tipo=reporte, appid=578080, usuario=palmerjoshua
[9] Insertado en Mongo: tipo=upvote, appid=105600, usuario=michael59
[10] Insertado en Mongo: tipo=post_creado, appid=105600, usuario=ghicks, comentario="Among hit name yes fund."
[11] Insertado en Mong

ServerSelectionTimeoutError: localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6a9cf7fb6d0cf22f7873e651, topology_type: Single, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>